# 14 — Tucker Decomposition → PCA → Robust Mahalanobis Exploration

**Goal (exploration only — no clustering yet):**

Instead of treating the 5 MRI image types as independent blocks and compressing each
separately (notebook 12/13), Tucker decomposition treats them as one **3D tensor** and
decomposes all three modes simultaneously:

| Mode | Dimension | Meaning |
|------|-----------|--------|
| Mode 1 | 416 | patients |
| Mode 2 | 8,100 | HOG features |
| Mode 3 | 5 | image types |

By setting `rank[2]=1` the image-type mode is fully collapsed — the decomposition
learns one joint representation across all 5 views rather than treating them separately.

**Pipeline:** Raw HOG tensor → Tucker decomposition → PCA → check Mahalanobis safety.

Three HOG-rank values are tested: 1,000 (conservative), 1,600 (medium), 2,000 (aggressive).  
For each, PCA is applied at 70%, 80%, 90% to find a safe final dimensionality (< 138 components).

## Step 1 — Install dependencies

In [1]:
import subprocess, sys

def run_pip(*args):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet"] + list(args)
    r   = subprocess.run(cmd, capture_output=True, text=True)
    tag = args[-1]
    if r.returncode == 0:
        print(f"  OK    {tag}")
    else:
        print(f"  FAIL  {tag}")
        print(r.stderr[-400:] if r.stderr else "(no stderr)")

for pkg in ["tensorly", "scikit-learn"]:
    run_pip(pkg)
print("Done.")

import importlib.metadata
print(f"tensorly version: {importlib.metadata.version('tensorly')}")

  OK    tensorly
  OK    scikit-learn
Done.
tensorly version: 0.9.0


## Imports

In [2]:
import os
import time
import numpy as np
import pandas as pd
import tensorly as tl
from tensorly.decomposition import tucker
from sklearn.decomposition import PCA

print("All imports successful.")

All imports successful.


## Step 2 — Build the 3D tensor

Load `raw_hog_features_per_image.csv`, split the 40,500 HOG columns into 5 blocks of 8,100,
and stack them into a tensor of shape **(416, 8,100, 5)**.

In [3]:
NB_FOLDER = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NB_FOLDER, '..', '..'))
DATA_PIPELINE_DIR = os.path.join(PROJECT_ROOT, 'notebooks', '01_data_pipeline')
OUTPUT_DIR = NB_FOLDER
NB_DIR = DATA_PIPELINE_DIR      # legacy alias: shared pipeline inputs (pca_*.csv, clinical xlsx)
NB_PADDED = OUTPUT_DIR          # legacy alias: this notebook's own outputs

HOG_LEN     = 8_100
IMAGE_TYPES = ["cor", "gfc_sag", "gfc_tra", "masked_tra", "sbj_sag"]

print("Loading raw_hog_features_per_image.csv ...", end=" ", flush=True)
df          = pd.read_csv(os.path.join(NB_DIR, "raw_hog_features_per_image.csv"))
patient_ids = df["patient_id"].values
X_flat      = df.drop(columns=["patient_id"]).to_numpy(dtype=np.float64)  # (416, 40500)
print(f"done  shape={X_flat.shape}")

# Split into 5 image blocks and stack along new axis
blocks = [X_flat[:, i * HOG_LEN:(i + 1) * HOG_LEN] for i in range(5)]
tensor = np.stack(blocks, axis=2)   # (416, 8100, 5)

print(f"Tensor shape      : {tensor.shape}  (patients × HOG features × image types)")
print(f"Memory estimate   : {tensor.nbytes / 1e6:.1f} MB")
print(f"NaN in tensor     : {np.isnan(tensor).sum()}")

# Pass tensor to tensorly backend
tensor_tl = tl.tensor(tensor)

Loading raw_hog_features_per_image.csv ... done  shape=(416, 40500)
Tensor shape      : (416, 8100, 5)  (patients × HOG features × image types)
Memory estimate   : 134.8 MB
NaN in tensor     : 0


## Step 3 — Tucker decomposition (corrected: image rank = 3 and 5)

**Finding from debugging:** `rank[2]=1` collapses all 5 image types into a single direction,
producing a degenerate patient representation where 99% of variance is in 1 component.

**Fix:** use `rank[2]=3` or `rank[2]=5` and represent each patient as the **flattened core**
`core.reshape(416, -1)`. With rank[2]=3 and HOG-rank=1000 this gives (416, 3000) with
all columns having non-zero variance, and PCA at 90% selects ~57 components (safely below 138).

| Rank | Core shape | Flat patient repr. | PCA 90% (expected) |
|------|-----------|-------------------|-------------------|
| (416, 500,  3) | (416, 500,  3) | (416, 1500) | ~30–45 |
| (416, 1000, 3) | (416, 1000, 3) | (416, 3000) | ~57 |
| (416, 1000, 5) | (416, 1000, 5) | (416, 5000) | ~88 |

In [13]:
ranks_to_try = [
    (416,  500, 3),   # conservative HOG, image rank 3
    (416, 1000, 3),   # medium HOG, image rank 3
    (416, 1000, 5),   # medium HOG, image rank 5 (all image types preserved)
]

tucker_results = {}   # rank_label → patient_repr np.ndarray

for rank in ranks_to_try:
    rank_label = f"hog{rank[1]}_img{rank[2]}"
    print(f"\n[Tucker rank={rank}]")
    print(f"  Decomposing tensor {tensor.shape} ...", end=" ", flush=True)
    t0 = time.time()

    core, factors = tucker(tensor_tl, rank=rank, random_state=42)

    elapsed = time.time() - t0
    print(f"done in {elapsed:.1f}s")

    core_np = tl.to_numpy(core)

    # Flatten core across HOG and image dimensions: (416, rank[1]*rank[2])
    # This correctly captures patient variation across both modes.
    patient_repr = core_np.reshape(416, -1)

    nan_count  = np.isnan(patient_repr).sum()
    n_nonzero  = (np.var(patient_repr, axis=0) > 1e-10).sum()
    print(f"  Core shape              : {core_np.shape}")
    print(f"  Patient repr (flat)     : {patient_repr.shape}")
    print(f"  Non-zero variance cols  : {n_nonzero} / {patient_repr.shape[1]}")
    print(f"  NaN values              : {nan_count} {'OK' if nan_count == 0 else 'WARNING'}")

    tucker_results[rank_label] = patient_repr

print("\nAll Tucker decompositions complete.")


[Tucker rank=(416, 500, 3)]
  Decomposing tensor (416, 8100, 5) ... done in 30.8s
  Core shape              : (416, 500, 3)
  Patient repr (flat)     : (416, 1500)
  Non-zero variance cols  : 1500 / 1500
  NaN values              : 0 OK

[Tucker rank=(416, 1000, 3)]
  Decomposing tensor (416, 8100, 5) ... done in 26.0s
  Core shape              : (416, 1000, 3)
  Patient repr (flat)     : (416, 3000)
  Non-zero variance cols  : 3000 / 3000
  NaN values              : 0 OK

[Tucker rank=(416, 1000, 5)]
  Decomposing tensor (416, 8100, 5) ... done in 40.9s
  Core shape              : (416, 1000, 5)
  Patient repr (flat)     : (416, 5000)
  Non-zero variance cols  : 5000 / 5000
  NaN values              : 0 OK

All Tucker decompositions complete.


---

## Debug — Diagnose why PCA gives only 1 component

Run all four checks on the first Tucker result (rank 1000) before proceeding.

### Check 1 — Core and factor shapes + variance of core[:, :, 0]

In [8]:
# Re-run Tucker at rank (416, 1000, 1) and inspect every output
print("Re-running Tucker (416, 1000, 1) for diagnosis ...")
core_d, factors_d = tucker(tensor_tl, rank=(416, 1000, 1), random_state=42)
core_d_np = tl.to_numpy(core_d)

print("\n=== Core and factor shapes ===")
print(f"  Core        : {core_d_np.shape}")
for i, f in enumerate(factors_d):
    f_np = tl.to_numpy(f)
    print(f"  factors[{i}]  : {f_np.shape}  "
          f"min={f_np.min():.4f}  max={f_np.max():.4f}  std={f_np.std():.6f}")

print("\n=== Variance of core[:, :, 0]  (current patient_repr approach) ===")
pr_current = core_d_np[:, :, 0]   # (416, 1000)
print(f"  Shape                 : {pr_current.shape}")
print(f"  Global min/max/std    : {pr_current.min():.6f} / {pr_current.max():.6f} / {pr_current.std():.6f}")
print(f"  Per-column variance   : min={np.var(pr_current, axis=0).min():.8f}  "
      f"max={np.var(pr_current, axis=0).max():.8f}  "
      f"mean={np.var(pr_current, axis=0).mean():.8f}")
print(f"  Non-zero variance cols: {(np.var(pr_current, axis=0) > 1e-10).sum()} / {pr_current.shape[1]}")

# Quick PCA to confirm
from sklearn.decomposition import PCA as _PCA
_pca = _PCA(n_components=0.90, random_state=42)
_pca.fit(pr_current)
print(f"\n  PCA 90% on core[:,:,0] selects {_pca.n_components_} component(s)")

Re-running Tucker (416, 1000, 1) for diagnosis ...

=== Core and factor shapes ===
  Core        : (416, 1000, 1)
  factors[0]  : (416, 416)  min=-0.1800  max=0.2236  std=0.049029
  factors[1]  : (8100, 1000)  min=-0.1076  max=0.9996  std=0.011111
  factors[2]  : (5, 1)  min=0.3863  max=0.4911  std=0.046228

=== Variance of core[:, :, 0]  (current patient_repr approach) ===
  Shape                 : (416, 1000)
  Global min/max/std    : -28.154671 / 510.122806 / 0.822921
  Per-column variance   : min=0.00000000  max=624.03782601  mean=0.67557319
  Non-zero variance cols: 416 / 1000

  PCA 90% on core[:,:,0] selects 1 component(s)


### Check 2 — Try image rank = 3 and 5

In [9]:
for img_rank in [1, 3, 5]:
    print(f"\n[Image rank = {img_rank}]  Tucker (416, 1000, {img_rank})")
    core_t, factors_t = tucker(tensor_tl, rank=(416, 1000, img_rank), random_state=42)
    core_t_np = tl.to_numpy(core_t)

    # Flatten the core across HOG and image dimensions → (416, 1000 * img_rank)
    core_flat = core_t_np.reshape(416, -1)

    print(f"  Core shape              : {core_t_np.shape}")
    print(f"  Flattened core shape    : {core_flat.shape}")
    print(f"  Flattened core std      : {core_flat.std():.6f}")
    print(f"  Per-col variance range  : "
          f"[{np.var(core_flat, axis=0).min():.8f}, "
          f"{np.var(core_flat, axis=0).max():.8f}]")
    n_nonzero = (np.var(core_flat, axis=0) > 1e-10).sum()
    print(f"  Non-zero variance cols  : {n_nonzero} / {core_flat.shape[1]}")

    _p = _PCA(n_components=0.90, random_state=42)
    _p.fit(core_flat)
    print(f"  PCA 90% → {_p.n_components_} components")


[Image rank = 1]  Tucker (416, 1000, 1)
  Core shape              : (416, 1000, 1)
  Flattened core shape    : (416, 1000)
  Flattened core std      : 0.822921
  Per-col variance range  : [0.00000000, 624.03782601]
  Non-zero variance cols  : 416 / 1000
  PCA 90% → 1 components

[Image rank = 3]  Tucker (416, 1000, 3)
  Core shape              : (416, 1000, 3)
  Flattened core shape    : (416, 3000)
  Flattened core std      : 0.544907
  Per-col variance range  : [0.00612663, 624.00312706]
  Non-zero variance cols  : 3000 / 3000
  PCA 90% → 57 components

[Image rank = 5]  Tucker (416, 1000, 5)
  Core shape              : (416, 1000, 5)
  Flattened core shape    : (416, 5000)
  Flattened core std      : 0.447001
  Per-col variance range  : [0.00459151, 623.98865685]
  Non-zero variance cols  : 5000 / 5000
  PCA 90% → 88 components


### Check 3 — Alternative patient representation: project original tensor onto Tucker factors

Instead of using just the core, project the original tensor onto the Tucker factor matrices
for the HOG and image-type modes. This gives the coordinates of each patient in the Tucker basis.

```
patient_repr[p, i, j] = Σ_h Σ_t  tensor[p, h, t] * factors[1][h, i] * factors[2][t, j]
```

With image rank=1, j has one value → result can be squeezed to (416, 1000).

In [10]:
f1_np = tl.to_numpy(factors_d[1])   # (8100, 1000)  HOG factor
f2_np = tl.to_numpy(factors_d[2])   # (5, 1)        image-type factor

# ── Approach A: tensordot as suggested ────────────────────────────────────────
# Contract core's image dim (axis 2, size 1) with factors[2]'s second dim (axis 1, size 1)
pr_A = np.tensordot(core_d_np, f2_np, axes=([2], [1]))  # → (416, 1000, 5)
pr_A = pr_A.reshape(416, -1)                             # → (416, 5000)
print("Approach A — tensordot(core, factors[2], axes=([2],[1])) reshaped:")
print(f"  Shape  : {pr_A.shape}")
print(f"  std    : {pr_A.std():.6f}")
print(f"  non-zero var cols: {(np.var(pr_A, axis=0) > 1e-10).sum()} / {pr_A.shape[1]}")
_p = _PCA(n_components=0.90, random_state=42); _p.fit(pr_A)
print(f"  PCA 90% → {_p.n_components_} components\n")

# ── Approach B: project ORIGINAL tensor onto Tucker HOG and image factors ─────
# Step 1: contract mode-3 (image types) with f2  →  (416, 8100, 1)
T_proj3 = np.tensordot(tensor, f2_np, axes=([2], [0]))
# Step 2: contract mode-2 (HOG)       with f1  →  (416, 1000)
pr_B = np.dot(T_proj3[:, :, 0], f1_np)
print("Approach B — project original tensor onto Tucker HOG + image factors:")
print(f"  Shape  : {pr_B.shape}")
print(f"  std    : {pr_B.std():.6f}")
print(f"  non-zero var cols: {(np.var(pr_B, axis=0) > 1e-10).sum()} / {pr_B.shape[1]}")
_p = _PCA(n_components=0.90, random_state=42); _p.fit(pr_B)
print(f"  PCA 90% → {_p.n_components_} components\n")

# ── Approach C: project original tensor, keep all 5 image directions ──────────
# Re-run Tucker with image rank=5 to keep all image-type dimensions
print("Re-running Tucker (416, 1000, 5) for approach C ...")
core_5, factors_5 = tucker(tensor_tl, rank=(416, 1000, 5), random_state=42)
f1_5 = tl.to_numpy(factors_5[1])   # (8100, 1000)
f2_5 = tl.to_numpy(factors_5[2])   # (5, 5)

T_proj3_5 = np.tensordot(tensor, f2_5, axes=([2], [0]))  # (416, 8100, 5)
pr_C = np.einsum('pht,hi->pit', T_proj3_5, f1_5).reshape(416, -1)  # (416, 5000)
print("Approach C — project tensor, image rank=5, flatten HOG×image:")
print(f"  Shape  : {pr_C.shape}")
print(f"  std    : {pr_C.std():.6f}")
print(f"  non-zero var cols: {(np.var(pr_C, axis=0) > 1e-10).sum()} / {pr_C.shape[1]}")
_p = _PCA(n_components=0.90, random_state=42); _p.fit(pr_C)
print(f"  PCA 90% → {_p.n_components_} components")

Approach A — tensordot(core, factors[2], axes=([2],[1])) reshaped:
  Shape  : (416, 5000)
  std    : 0.368021
  non-zero var cols: 2080 / 5000
  PCA 90% → 1 components

Approach B — project original tensor onto Tucker HOG + image factors:
  Shape  : (416, 1000)
  std    : 0.822542
  non-zero var cols: 416 / 1000
  PCA 90% → 291 components

Re-running Tucker (416, 1000, 5) for approach C ...
Approach C — project tensor, image rank=5, flatten HOG×image:
  Shape  : (416, 5000)
  std    : 0.446966
  non-zero var cols: 5000 / 5000
  PCA 90% → 309 components


## Step 4 — PCA on each Tucker representation

PCA at three variance thresholds (70%, 80%, 90%) is applied to each Tucker output.
The goal is to find combinations that give a final component count below **138**
(the Robust Mahalanobis safety limit = 416 patients / 3).

In [14]:
PCA_THRESHOLDS = [0.70, 0.80, 0.90]
SAFE_LIMIT     = len(patient_ids) / 3   # 138.67

pca_results = []

for rank_label, patient_repr in tucker_results.items():
    print(f"\nTucker {rank_label}  (patient_repr shape: {patient_repr.shape})")
    print(f"  {'PCA threshold':<16} | {'N components':>14} | {'Status':>8}")
    print(f"  {'─'*44}")

    for thresh in PCA_THRESHOLDS:
        pca   = PCA(n_components=thresh, random_state=42)
        X_pca = pca.fit_transform(patient_repr)
        n     = X_pca.shape[1]
        safe  = "SAFE  " if n < SAFE_LIMIT else "UNSAFE"
        print(f"  {thresh*100:.0f}%{'':<13} | {n:>14} | {safe:>8}")

        pca_results.append({
            "tucker_rank":      rank_label,
            "pca_threshold":    f"{thresh*100:.0f}%",
            "final_components": n,
            "safe_mahalanobis": n < SAFE_LIMIT,
        })

df_pca = pd.DataFrame(pca_results)
print("\nPCA exploration complete.")


Tucker hog500_img3  (patient_repr shape: (416, 1500))
  PCA threshold    |   N components |   Status
  ────────────────────────────────────────────
  70%              |              1 |   SAFE  
  80%              |              1 |   SAFE  
  90%              |             23 |   SAFE  

Tucker hog1000_img3  (patient_repr shape: (416, 3000))
  PCA threshold    |   N components |   Status
  ────────────────────────────────────────────
  70%              |              1 |   SAFE  
  80%              |              1 |   SAFE  
  90%              |             57 |   SAFE  

Tucker hog1000_img5  (patient_repr shape: (416, 5000))
  PCA threshold    |   N components |   Status
  ────────────────────────────────────────────
  70%              |              1 |   SAFE  
  80%              |              2 |   SAFE  
  90%              |             88 |   SAFE  

PCA exploration complete.


## Step 5 — Summary table

All combinations flagged SAFE or UNSAFE for Robust Mahalanobis.

In [15]:
print(f"Safe limit: n_components < {SAFE_LIMIT:.2f}  (= {len(patient_ids)} patients / 3)")
print()

print(f"{'Tucker rank':<16} | {'PCA 70%':>14} | {'PCA 80%':>14} | {'PCA 90%':>14}")
print("─" * 65)

for rank_label in df_pca["tucker_rank"].unique():
    subset = df_pca[df_pca["tucker_rank"] == rank_label]
    print(f"{rank_label:<16} | ", end="")
    for _, row in subset.iterrows():
        n    = int(row["final_components"])
        flag = "SAFE  " if row["safe_mahalanobis"] else "UNSAFE"
        print(f"{n:>4} ({flag})  | ", end="")
    print()

print("\nRecommended combinations (SAFE for Robust Mahalanobis):")
print(f"  {'Tucker rank':<16} | {'PCA threshold':>14} | {'Final components':>18}")
print("  " + "─" * 55)
safe_rows = df_pca[df_pca["safe_mahalanobis"]].sort_values(["tucker_rank", "pca_threshold"])
if len(safe_rows) == 0:
    print("  None found.")
else:
    for _, row in safe_rows.iterrows():
        print(f"  {row['tucker_rank']:<16} | {row['pca_threshold']:>14} | {int(row['final_components']):>18}")

Safe limit: n_components < 138.67  (= 416 patients / 3)

Tucker rank      |        PCA 70% |        PCA 80% |        PCA 90%
─────────────────────────────────────────────────────────────────
hog500_img3      |    1 (SAFE  )  |    1 (SAFE  )  |   23 (SAFE  )  | 
hog1000_img3     |    1 (SAFE  )  |    1 (SAFE  )  |   57 (SAFE  )  | 
hog1000_img5     |    1 (SAFE  )  |    2 (SAFE  )  |   88 (SAFE  )  | 

Recommended combinations (SAFE for Robust Mahalanobis):
  Tucker rank      |  PCA threshold |   Final components
  ───────────────────────────────────────────────────────
  hog1000_img3     |            70% |                  1
  hog1000_img3     |            80% |                  1
  hog1000_img3     |            90% |                 57
  hog1000_img5     |            70% |                  1
  hog1000_img5     |            80% |                  2
  hog1000_img5     |            90% |                 88
  hog500_img3      |            70% |                  1
  hog500_img3      |     